# PA3 Section 2.1 Pendulum SAC — Visualization Notebook

Training is done by scripts. This notebook only loads logs, creates plots/tables, and generates rollout videos/GIFs.

## Checklist

- Evaluation starts at timestep 0.
- Evaluation is deterministic/greedy.
- Each evaluation point averages 20 episodes.
- Pendulum uses 500 initial random-action steps.
- CI is over seeds, not episodes.
- Default torque limits are unchanged.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUTPUT_ROOT = Path('../outputs/pendulum_final_300k_clean')
PLOTS_DIR = OUTPUT_ROOT / 'plots'
VIDEOS_DIR = OUTPUT_ROOT / 'videos'
OUTPUT_ROOT, PLOTS_DIR, VIDEOS_DIR

## Load combined logs

In [ ]:
import sys
sys.path.append('..')
from pendulum_sac_lib import load_all_eval_logs, load_all_train_logs

eval_df = load_all_eval_logs(OUTPUT_ROOT)
train_df = load_all_train_logs(OUTPUT_ROOT)
print(eval_df.shape, train_df.shape)
eval_df.head()

## Select manual alphas after manual-search stage

In [ ]:
# Run this after scripts/run_02_manual_search.sh if you want to recompute selection inside notebook.
!python ../select_manual_alphas.py --output-root {OUTPUT_ROOT} --tuning-seeds 0,1,2

selected = json.loads((OUTPUT_ROOT / 'selected_manual_alphas.json').read_text())
selected

## Generate required plots

In [ ]:
# Raw required plots
!python ../make_pendulum_plots.py --output-root {OUTPUT_ROOT} --smooth-window 1

# Supplementary smoothed plots for readability
!python ../make_pendulum_plots.py --output-root {OUTPUT_ROOT} --smooth-window 3

## Display key plots

In [ ]:
from IPython.display import Image, display
for name in [
    'auto_all_targets_eval_return_mean.png',
    'manual_vs_auto_panels_eval_return_mean.png',
    'reward_scaling_panels_eval_return_mean.png',
    'auto_all_targets_mean_abs_angle_error_deg.png',
]:
    p = PLOTS_DIR / name
    if p.exists():
        print(p)
        display(Image(filename=str(p)))

## Summary tables

In [ ]:
for name in ['manual_alpha_selection_table.csv', 'summary_auto_final.csv', 'summary_manual_vs_auto_final.csv', 'summary_reward_scaling_final.csv']:
    p = OUTPUT_ROOT / name
    if p.exists():
        print('\n', name)
        display(pd.read_csv(p).head(20))

## Generate rollout videos/GIFs

In [ ]:
!python ../make_pendulum_videos.py --output-root {OUTPUT_ROOT} --include-manual --include-scaling --ext gif --checkpoint best_model.pt

## Display a sample GIF

In [ ]:
from IPython.display import Image, display
gifs = sorted(VIDEOS_DIR.glob('*.gif'))
print('num gifs:', len(gifs))
if gifs:
    display(Image(filename=str(gifs[0])))